# GwenLand glcuda Wave 120 - repaired T4 event profile

Whole-prefill GPU timing with post-workload detailed event drain.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave120-in-process-stability-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
SOURCE_REV = "80eef8819aab801e2290bf58f9c87cf20fbf918f"
PATCH_SHA256 = "688b1740744a9dc810a5a64f6bd87f080b21de00cfab009fc8d28bb90404450b"
PATCH_GZIP_B64 = """H4sIAMUCpmoC/7Vb61LcSJb+z1Ok2QlG1VSJut9o3I1t7PWOMQzg7olwO0SWlKI06FLWhctgIvYh9gn3SfackykpVVKB3THLD6DycjLzXL9zMsvxXJd1OldeyvjelW9nDt8TdzxY+SLZu+U3otebWl5oreLIFkliJSlfeL6X3ptxwhY/OmMrFLfM9XzBgsgRrNftjofDLS90xB3rfuePaS4GA+EMBvaQO6PFoG/37JmYjCeDHh8N7aE9607dwcJxx1udToftOeJmL8x8f2t3d/dP7PjXX1mn2+6y3V67NxqyX3/d2t3be8F+h2kM5jEv7Kh5DP46mZ16UcgKEmzJ4xA6TZom556E8vx+m8VZGIq4zV5/enMI872Ax/fMjsJU3KVtxkOHcd+PbE5EuZ+KOOSpYOlSSFKxSLkXCoeGOsIVcSycTiwSz8m4z1Y8XSYmex0FAczX9uf6/CphPBYsyVYr3xOOpLe4R9rMhlVFzBbCjWBIfj44VJzuqwFZAvSTWy+1l8xLmLgDKjZo0VLEAg67tZslggGzgcB8LsIr2KWVxtxL5/OHd/4RNbTZ+xC2/D5cZenjfjEF5EOD8J/XUeh6V20mP8lp5VDc2Xz+jv7Kvn1cGhiYpOz07OT49ML69PH9xZztJGnMDtj2seBJFiMHYdOOAIYGXuglqWez5D5JRUBiDFYpHDEWLujNvcmO4HDAZraMblkaXQsQOY+RRT6IPxVXwKqAp7F3x4LMTz3khJQY7BLYBioAEgpEEMX3bXYhwiSKQSYgJSli17uDfp9nIfDySkSBSGNYdXs/P8nfPx2eXRxdnM+BoPcvAecYdYvO3w/Pjj+dnlunR2cW/KuNKYYc/eP06PXF0RtLseTi5G9HHzVq/eGQ+OaGcB6QheE5CbDsczbof2mxzktNTOxha5fBT70Ff4g5Fs2GX2YaWTfCNlrtckTA7yxwAhaNhGE9rQ+4vxIxT0E+c9Y1u3pXtLKu52y43rbCgbOR1hqLleCptRIhmMs9LGDqS5hmufH5HAyGg8CMlhzwuLX7qNgAdmnxODBkh1Rf4IiuhYqqxirVYoNMPQfMdM4WUeSr1ijmtg9NMBBaiKtnIoHVf3bHwzZ7Fd397Nyj43DAXOI4iufzI/zz8mXOYLkLMxGptRCgKuArri2yect1Qys3eqNYv/XLvpzpi5RFIKmDnIaHTDAKWbeKkZ6LA01pAkpI7MXBBg1i377R8ELsJgfvCdovjBbO+iwP/UVXEfBYWRwyOJsBzgXM5oVRduLPtpzEAi+BXns5BzcVHDw8ssqmsCH/b/7LI/ofYafCOfj88Phlu10lCacqmMIe2HbxYZvBTD+hxtyVQtva9BpHGvoLFqz30WG0xpamjOA8IqPV2i/UD/+cXBtyQQGB0reCpFWqZQA7NHTdAdv6HtVBBQhAAXgMXv9AjQxv5nNsMFpmcu2tjF5LUxeKTzAUB2g7DiEuGfoRomsrio1tiAZXoNxPBVN2fPLm6AP7zL8tvmxXVJOwgFxKLaB1xuJGxAn2kzaI5IWB41HTHAg1LnLgHFymsb3YbukTYQOINKwotL9nthouaVSoYDA40KOJKX2MMa0wDNh7tcpgpO4k5nOIkEvLpihm6CFNt4lECGcu9zHs697yKTtXE1wO+pvPecx3BDsB7fKAlb9oLX7EHYska+zQn4oYQHlh9zgOVoVey17CIXckB3QPgYbui1Da+AbP8EMWn0dcwD1gLilzYs9N5wxMHBZ4eKyZc77+WnvzVr7T9KqStJUkS2Axn0PYKjTTrvHXfpa/0hVY0hFLAqUX3qm44XI4DP2pnKgdoHA4WpvrxUmjeUqklJO8EiGGWcAcYSSd6HblGMkyS53oNjRKUwDBaS4UoNBn0rs2S+Os6t9xpIUjAKivQRN9GCmGCrI7oHNthhxol4u01W6LnWkushDX3h47WSQiviG41YlC/x5CHKAntopAwrQZhdRnJvubECsWAfgG+wUrgoEw7UYUpPQDgmQEOg44COLdAgJHgNgg2BAQ93MfQ7wBK/YhBBXUvLSSEVzDDAg1DP0BqgDswef3XnhF9PvdbiehjERLHYA/ZmF0FXemMVI6qTxWgLo0cRWl1MhQnJ3CvglySuuvysgsenW9oh7glgV7D41v6TdWBKz1UUoDpRT63Xyv2qpZyG+45/MFel99c5APhakf1tzFZwoz/W5H8QT07+GP7UqE/mN7/vDY/mN7GSWABQv2YPPcnGEPnPT5jnSlesbYo+yQVqAFHr/bNZUrrXUUbACDTnnDgNLZ8oQBVGQ/YdoM6JjtPT+5jj72q5YKynYllbwgRU3Juq1ukEVVHjRVSiPkgQAe/bH98PjHNvAu36DGaEwyC0Et7lORQGzjTt4ScDvZwGWKmbiWicts7NzAlHIA7WBjb7klMwtvY75CRe62No7HDT81slV3ZHlgrnjctbgJeHBDoAoxmiJI051x/puavtSBlIPjtSHFH/LklfgDCAkGg+/JMdhDSaTAzWoTjzqQCG/yksXadMBoGuLm28U01MWvGY9Tmo6xI89512OLsYoSD2lrsaKFc2i/FNctD9wzoFoRZgEFOjCeNXX+Lq/5dCzKycAsecwfSjH2f9S8elMVIKR9lTzWrUyxMDehnFX5Z9iqPrrRYzb6xD/tD8mDFVtt6FQ7bugp5FzvgnM0rbPB9z7pf59zsU9OrLnXDWau/jQIuRQumIjj2WmH4OB3yLgQmNSKxFpBfiAFrOTYIFy5YUVtDwlWRdYoqtwSG5qAU319eoMEtKHDp5v13KXmEKUXxEzYaahYJ7G9pwDWHgDvSm263qcKzov+0J0tZtwe98aDWXfhLKZd+OAOBv3FpDdYiKkzs6fuyDTH0+lw5tpObzrh9nDqDkZ94dj2WExG0DIczEZdMex1eV7QxrrzE3urVqIb+rHmPG6P2e64PcGKM8PSLAE/EDJ6wVseO2zFAZAa4BKxEUJV2jK32BbWHGWC77refG5bN5HnqIIpNSf3oQ2JfxoFHvx9OKR/XmGhip2gCwVICt5JEcoLuO98qihgO26u3x/Bznb7fblBtsoWQDwGqAsgG49yLrAuKOObCAIr7I0JVAkq6MiyGJNAeS9H6L25rGAzwe0lxJWw43oIst++/ciKijb6d0LMWChggKBF/NekJMV9jNf3HTfDIMXTFEA+xqGz4/Pdv0+JZSY7WaGZAe5MPV+H6Id7r8ytDpJqyra1Wl5Tt8bH9ZONDrFCvl6qx8rjvqxDg0Qhu6BjxdGtShXYgtvXGq3bpSdL7beQlmMWAttl/ApoJhAzAQ0IAeYnxwOVxCKqTr5xEtts0O71QG6zYbs/e05wrgX8gyMira/XQ6zoLMRcjYONPYIyUK1UJJF/I5pKkSq4gzCsr1Prys+qNVER3nhxFAYgJEuEmAA4lf6y/BGBe4w9B6aD8EBWP+Ool3khFT/kEV5fjO3sNNDQMFrDBlS1jXngVes8QUH81/sLEpUIFsJxQKCnF/8goSpGMIQ7lTwUVA7Sw8CajHbp6qYgldJdQAdtDKtgGaR/IGAsIwDZNIqYcfnuA94NWR9PrOPjw4PeJYtWkFhCBtlGtSlJoTYUZ8W0MlcN3Cqu3WXOasjZu6Pj45ZSE5Q+CBDXM+jWhe28hj96gbHgAF7ByKqijqXOhe/O51T3oEKX1IGc00S0zT5Czt2qZe577ANMo+0phU8yAG6UIDPOQh6jLRRHksl9TrnI7Etq4EBMdolrXSITEb13YFQH/8lz+NzSNbmDhHzPvleJ9n98diLbWHog17D1RYXtkkuNhyy5ofFQC3V1u9iszhQHn2V/BX8qvTmg1YsSFDQZoJeIWCujEwSrJBcsP0UmaEbA/xnFbVZt8wDXt9bmBgGXSBeIvAQqkzYbtdDGynryDQfGJMZ2RWu3W6aXWODQJRBHTzSYTttTtjvsTtAhQUuzuZHKKPi8VYFeKGdtzL702/lO6yxnB+WIJjex8Qhvjt4enVkQg6yzo/P3bz4dfpDHSbD42lorpTSt+7x/bNrTGqj8k9tbo/KEJlZrBBWxa8GkcnewtpPDiwvYxMnv51X+rGvQ0Cqj8gE2lIOflAORB2UaPkc+Fldf9TVI34a9GQKW4bAnActmddMhS7upGe/vGkcXAKddVbU6158Viw4nZOG5Pqa1tg1NTOsb/MonWMz38Oaq0lOVhwQJvUlviLzqTQbjdm+6mVsEcJYCXGusobW8vkfQLGEBv1eoDsZ5cRXMccchQKcFxBzZNUO3Sthq0GYVw42dBOJSDRrIqxbfNeszyW8aOf6dz8+Ez+8IDNSi1rl885CDVohQ6a0QEN8BVy8hqHj/AjspwhbDWgRlVIlZ0CiJnZYxia7qY/UhRAxBzzASgogmuwDOFI9O1KMSBhlSwnhJLoHNQyyQr0r26ApiT70mIYRC7xEIciI0AKTMIjevbXeu/GjB/ZKYHiSDLKVDPB8mn72eJtm0WQXvtXQL3CCitbpykgJoMmjsui9XpMt8ppSnchgIXQudKzjbnwD4cETnChyigm+URqhHMU7s3WC2wSLbzlY8tO/Z1wyQnimx9WQwwZStP5l0271RDq4RH0crYaW4pQSLV+rZRZstgf+W4wVFgxuLr9aCJwAL3EGf1Nf4Tdg/w4eXbfYbxDuQKGV6Sc4vyuoySADn858wOVPSwTFKLLCB/FY2fx9U9/4WeNT8BY+FdX1gKUjbusJbomr1DMxQxOkL48UTYa1SXiT0pwfKnMKzBOTvPz9fbYCuSbGl1UjmxXfuQ7tt/WE6lfpsw34o49D1EVMKL8DafJAliCtsHx9riTtup75URy2/SErnXJJR20nY//73/4AnodtvebNauCwkY67SO/kkDx0E+QYJH7XMU11ZYYZJmQV7d/rJ3FiI8b1FrQAj21ThZTZxxWwx5e7EnQ0Hg7Ho9YdDseh3Z6NF15mOJsNxfyrskWmOur1Fd9wfOP0R702mXTGb2kPb6c4Gbm88c2b90WI2W4xHGwsvat1awUW1o+GOemi28Ls3QKMli6KSGtjUw7n8D4Qm/5F391gf0TLntWv9gm9v6V3X2cd3dMFPKQvePKLH9xa+WgYYup+nLQcs9QLRwdHC0YKdfB+gUoVsPMwzBVzk1TMJEsaovybF28BqAMb3gU1hSe2HbknjG1Wa2JA/5bf2MkTBxvHxG74nXCvvlFGLotAlmsElXqguIkz44iChW1Q7EndeQrcAeFWrglNJLVdGGZFAG6vhq7I1gJAe+d1nwtbTzyyqKZqMHbgT2Y5cFHGH6qwUsLFsA/zCOhQm8QAJsqsl+3xZfRYCIfryi4wcsxkqYK9bQVv66FpwVOVCfMVCb1jKoFaiFLqMliWlvPCkaQRJvhm6lIClpKZuwQvhljItgU750PX/ByFUHz01FSMwBciv2Q+IU2uYQXWu32PnXGy4uLYw9zS+fcvXm8+lSPBJBQoIgC+qqpd63EfubecPSypXRbm8nj15XoPS5uaF71qYOKJHOuwSwd0lS9VLDnrFQyUUFJiqJBEQBCdAI2Cb4M8AeoPxVIH3q5NzOcRkh/mNPOMu6AOpb6665SuXyy/yNXCMpdRAixjyUhADlVoRjH4F3lLwoM0WYL0uaFvaESFsFsNO/hobqPGULSPfkZgKci0sMxfOWZmGMgt0cZvsRD3CQHeYgKMA0pytvJXwyRJuYXPM9iOwVIgYWmX1nl0C7JE7vaxRoyruKopTVcXF8RQx6aYZu1Ed7ChY0YtfVYGrkYFjg7/A0+fuYZWlZjVppIcZ6v6asm68lDfVxWbAV8a35BtLiivulplkQaUcodY6jfPkC5aFE2OGgOEbi5RYmDzPgoBiuvNPbqPPVNf/XpFXaMTAcbuZj7lZ7IEUKWtZCvtaytTDN9MhcqB4X8Ou+EpK2Etr1GDMjRdlCb3ZsaMshoUdSoPw2haZmrOyeGfT8ULXL6OLRgwfekixoiZKC9ACls7n3Sf4vKpf8ike47OQIGnoLqrJha/4TmHp1Qv8OV0C9D+VJ2UPikq73N2jVu9qFTcxQyps9CfDdq/7fSaiqvsh+S1jB5+7Felys4etcIvc3gF7Ta/06U6AHsF1mr1wAUyl3zB2cHrtCdLG8c8UXPGHKK6XyzBKykeX5lMOV696rVdKPdjNTlEU1Tox9IdCfhsiWzGfvsjge9f5FxLA0X5OvMD5Ql1zhm9UyAAhQjq3oM4VUhSI8VkZJO8+pdpe0pFPyVBxwQ5vvATh4mbALdP9GuYumhXsdkfDgTvuDWY2H3SHg9504rjTRb/XnYwWk8m4O+LOdDrgQ9MUI7fv8q7bGw1Gfdt2bOFy0e/2Zv3+xLHHfOJMJrPJYPN9Z7l0DXmXXbLYNCTwU6ivBqtPZSWpsIgyvDgxOlYJSpVPB6ek7l0WURY6+DUamc/gK1+8XpGOG51RSUfcCVu6MfnYhaKPLL3ANrFIovkQE8RLuiRTsoTfy8Ch4fQidFRusWSdLPJFJ6+Nkf/1wTUiQER3BeEwILhb+mjyz6AdXmyyTyGqlwbP34nwwguosqFoXrZlJBJ3tp85CrGTU7TxV4SvcLijnhuWlBzEhoA4tOUSFfKRwG0UX6MZguom5IzBMDAzKeCc7hsLpOxShlIC5TdRXnW6Fd7VMgUDeXd0/BvtiJYBzDRlkEJ48h0nS+wY81WJjsdjvCmYYVVFwwCr7JheqD9UjKmM6xCSBH5JSWBIXuK50X0gz012WdWs+bwp5NN7UPA3IlH4QGUjoA1C7jrMggXGKTJegvsxQQDSH1+4aYXYVQbgGPZhVj2fvczCa4o8puPdWLbwfOP07Ojt+w8frFeHF6//s1aNx7PQLcwtD/FtY24dFfhAfotkauEEo49vWvCXbYaWLM22ajH04jbSFBBdZsJWfpbIo4NqUDiRVrbCYIUGQh9rpGiJPTqcencr8welbZK270ESIW/hUW5Kk2u0UAlJAReUq2SLwEtBC/elG6AUEGuluBbD7yMVj3JR/WvUUA2w8oYVUMyT1zJb1KCORIb41a74hvtriKHO2l2GnD2/OHx3ZH08PD46V6+JdG7jBxJ164dvtmSOSPdl4G8rF2ZNZtB0jVNghvJhUiVeIwagJO6AdakYuRaftf6+6mfr/SvAs+iRGJYsDVXS1P+0XsJ06Ky+pFfPm5EKFcbiFl6ZoaHkeVnt2bgZCxsctrpd7lYfXJV3G9yOIyvOIK14odR1jUvGXxaZfQ1QQ9ytICX6CyGu/MMicu7nC1QE2BCAoAYOP7ttinDDcbfdx++PDid9/Od7ZPe4tf6xat7oVFH5pVclxQKnKn042ZYpIQrk9WSz5MsxBMmx7Ro1RO1k5SJ/EwwhHWdJS70VWHwMv2Yiw7p6HhrQ3TkRfhVunZ58DlEEnAAWuKVIlBDQoVuKW+6tZz3PcrSzzik83Y6R4JdlweXTQ9AdpYlNw9cXATCOq5gqGkMUMySdDXOL174w8nOSfoHJREdv098Am90WuAdYZb+Z3mNVtbH4z3YPWNj8VDg3NT3qwg7QY/y7rKnXkEOeF+Wh/OsKGkwJpTfHqJso2BHI79g6iHNqxHKvSu/ZTHZE0aQSGFCrcqxRYBEPMqEkqlGTl4qoaUmefCYilsUYrHfhNzUkDCyxjJdSerD2Dr/K0opGdBvY8rTe7f5Jndv9N+pbg67VVFDt2LR9wWOjSUnzqLAeTLY61bGa5jZ5sTpcYbV8LX9qm49RR62lAdKrTgeUOAxnI+1G/ymfis/z5zkz4f92fUj+peRVQ1+BERveAevK034y6uoMwXPCRv8PDZukt4JBAAA="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/kaggle/working/wave120")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave120-in-process-stability-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave120.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "67 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75.ptx",
                    "-o", ROOT / "wave120.cubin"], check=False)
    save("ptxas-sm75.log", resource)
    if resource.returncode or "spill stores" not in resource.stderr:
        raise RuntimeError("ptxas resource gate failed")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1"}

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1",
                "GLCUDA_TELEMETRY": "1"}

    phase = "event-profile"
    measured = run([exe, MODEL, "profile"], cwd=TREE, env=prod_env, check=False)
    save("event-profile.log", measured)
    if measured.returncode or "[wave120-profile]" not in measured.stdout:
        raise RuntimeError("single-pass production profile failed")

    profile = json.loads(re.search(r"\[wave120-profile\]\s*(\{[^\n]+\})", measured.stdout).group(1))
    stages = [json.loads(x) for x in re.findall(r"\[wave120-stage\]\s*(\{[^\n]+\})", measured.stdout)]
    if len(stages) != 8 or profile["gpu_prefill_ms"] <= 0:
        raise RuntimeError(f"event profile contract failed: {profile}, {len(stages)} stages")
    stage_sum = sum(x["total_ms"] for x in stages)
    summary = {"wave": 120, "gpu": fields, "model": model_meta,
               "profile": profile, "stages": stages, "stage_sum_ms": stage_sum,
               "stage_sum_over_gpu_total": stage_sum / profile["gpu_prefill_ms"],
               "retention_authority": False,
               "target_15000_tps_achieved": profile["gpu_prefill_tps"] >= 15000}
    (RESULTS / "wave120-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("WAVE120_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
